In [ ]:

import pandas as pd
import numpy as np
import os


# Air quality data (hourly)



- Load the raw hourly AQI file for 2023

- Fix timezone and keep `date` as the index

- Run a few simple QA checks (missing values, negatives, PM2.5 vs PM10, UV, time gaps)

- Create a clean daily dataset ready to merge with weather data


In [2]:

RAW_WEATHER_FILE = "../raw/weather_data.csv"
RAW_AQI_FILE = "../raw/air_quality.csv"

PROCESSED_DIR = "../processed"
REPORTS_DIR = "../reports"
PROCESSED_FILE = os.path.join(PROCESSED_DIR, "daily_aqi_processed.csv")
QA_REPORT_FILE = os.path.join(REPORTS_DIR, "qa_summary.csv")

YEAR = 2023
TIMEZONE = "Asia/Ho_Chi_Minh" 
print(f"Paths set for {YEAR} in {TIMEZONE}.")

Paths set for 2023 in Asia/Ho_Chi_Minh.


In [3]:
df_aqi_raw = pd.read_csv(
    RAW_AQI_FILE,
    parse_dates=['date'], 
    index_col='date'
)
print(f"Loaded {len(df_aqi_raw)} raw hourly rows.")
df_aqi_raw.head()

Loaded 8760 raw hourly rows.


,Unnamed: 0,pm10,pm2_5,uv_index,uv_index_clear_sky,nitrogen_dioxide,sulphur_dioxide,ozone,carbon_monoxide,carbon_dioxide
date,,,,,,,,,,
2022-12-31 17:00:00+00:00,0,95.0,65.0,0.0,0.0,84.40,37.50,25.0,993.0,NaN
2022-12-31 18:00:00+00:00,1,86.1,59.0,0.0,0.0,70.25,30.75,28.0,846.0,NaN
2022-12-31 19:00:00+00:00,2,83.4,57.0,0.0,0.0,65.80,28.90,26.0,821.0,NaN
2022-12-31 20:00:00+00:00,3,80.1,54.8,0.0,0.0,63.95,28.45,22.0,834.0,NaN
2022-12-31 21:00:00+00:00,4,69.4,47.5,0.0,0.0,60.10,27.20,20.0,838.0,NaN


In [4]:
# Convert timezone and keep date as index
df_aqi_local = df_aqi_raw.tz_convert(TIMEZONE)
print(f"Converted timezone to {TIMEZONE}.")
del df_aqi_local["Unnamed: 0"]
df_aqi_local.head()

Converted timezone to Asia/Ho_Chi_Minh.


,pm10,pm2_5,uv_index,uv_index_clear_sky,nitrogen_dioxide,sulphur_dioxide,ozone,carbon_monoxide,carbon_dioxide
date,,,,,,,,,
2023-01-01 00:00:00+07:00,95.0,65.0,0.0,0.0,84.40,37.50,25.0,993.0,NaN
2023-01-01 01:00:00+07:00,86.1,59.0,0.0,0.0,70.25,30.75,28.0,846.0,NaN
2023-01-01 02:00:00+07:00,83.4,57.0,0.0,0.0,65.80,28.90,26.0,821.0,NaN
2023-01-01 03:00:00+07:00,80.1,54.8,0.0,0.0,63.95,28.45,22.0,834.0,NaN
2023-01-01 04:00:00+07:00,69.4,47.5,0.0,0.0,60.10,27.20,20.0,838.0,NaN


In [5]:
# Keep only data for the selected year
df_aqi_2023 = df_aqi_local[df_aqi_local.index.year == YEAR]
df_aqi_2023.shape[0]  # should match raw rows for that year

8760

In [6]:
# Check how many missing values each pollutant has
pollutant_cols_1 = ["pm10", "pm2_5", "nitrogen_dioxide", "sulphur_dioxide", "ozone", "carbon_monoxide","carbon_dioxide"]
missing_counts = df_aqi_local[pollutant_cols_1].isna().sum()
missing_counts

pm10                   0
pm2_5                  0
nitrogen_dioxide       0
sulphur_dioxide        0
ozone                  0
carbon_monoxide        0
carbon_dioxide      8760
dtype: int64

In [7]:
# Drop carbon_dioxide because it is completely missing
df_aqi_local = df_aqi_local.drop(columns=['carbon_dioxide'])
print("Dropped carbon_dioxide column (all NaN)")
pollutant_cols = ["pm10", "pm2_5","nitrogen_dioxide", "sulphur_dioxide", "ozone", "carbon_monoxide"]

Dropped carbon_dioxide column (all NaN)


In [8]:
# Quick check: how many pollutants are missing per row
missing_pattern = df_aqi_local[pollutant_cols].isna().sum(axis=1)
print("Missing pollutants per row:")
print(missing_pattern.value_counts().sort_index())  # all rows have full data here

Missing pollutants per row:
0    8760
Name: count, dtype: int64


In [9]:
# Add a simple flag for any missing pollutant value
df_aqi_local['flag_missing_pollutants'] = df_aqi_local[pollutant_cols].isna().any(axis=1)
print(f"Flagged {df_aqi_local['flag_missing_pollutants'].sum()} rows with missing pollutants")
df_aqi_local[df_aqi_local['flag_missing_pollutants']].head()

Flagged 0 rows with missing pollutants


,pm10,pm2_5,uv_index,uv_index_clear_sky,nitrogen_dioxide,sulphur_dioxide,ozone,carbon_monoxide,flag_missing_pollutants
date,,,,,,,,,


In [10]:
pollutant_numeric_cols = ["pm10", "pm2_5","nitrogen_dioxide", "sulphur_dioxide", "ozone", "carbon_monoxide"]
negative_checks = (df_aqi_local[pollutant_numeric_cols] < 0 ).sum()
negative_checks  # => no negative values found

pm10                0
pm2_5               0
nitrogen_dioxide    0
sulphur_dioxide     0
ozone               0
carbon_monoxide     0
dtype: int64

In [11]:
# Show any rows that have negative values
df_negative_values = df_aqi_local[(df_aqi_local[pollutant_numeric_cols] < 0).any(axis=1)]
print(f"Found {len(df_negative_values)} rows with negative pollutant values")
df_negative_values[pollutant_numeric_cols].head(10)

Found 0 rows with negative pollutant values


,pm10,pm2_5,nitrogen_dioxide,sulphur_dioxide,ozone,carbon_monoxide
date,,,,,,


In [12]:
# Look at each column that could have negative values
for col in pollutant_numeric_cols:
    neg_count = (df_aqi_local[col] < 0).sum()
    if neg_count > 0:
        print(f"\n{col}: {neg_count} negative values")
        print(f"  Min value: {df_aqi_local[col].min()}")
        print("  Sample negative rows:")
        print(df_aqi_local[df_aqi_local[col] < 0][col].head())

In [13]:
# Add a flag if any pollutant value is negative
df_aqi_local['flag_negative_values'] = (df_aqi_local[pollutant_numeric_cols] < 0).any(axis=1)
print(f"Flagged {df_aqi_local['flag_negative_values'].sum()} rows with negative values")
df_aqi_local[df_aqi_local['flag_negative_values']].head()

Flagged 0 rows with negative values


,pm10,pm2_5,uv_index,uv_index_clear_sky,nitrogen_dioxide,sulphur_dioxide,ozone,carbon_monoxide,flag_missing_pollutants,flag_negative_values
date,,,,,,,,,,


- Check whether PM2.5 > PM10

In [14]:
# Check if PM2.5 is ever larger than PM10
pm_invalid = df_aqi_local["pm2_5"] > df_aqi_local["pm10"]
print(f"Found {pm_invalid.sum()} rows where PM2.5 > PM10")
df_aqi_local[pm_invalid][["pm10", "pm2_5"]].head(10)

Found 4 rows where PM2.5 > PM10


,pm10,pm2_5
date,,
2023-12-14 07:00:00+07:00,36.0,37.7
2023-12-14 08:00:00+07:00,33.8,39.5
2023-12-14 09:00:00+07:00,27.5,32.0
2023-12-14 10:00:00+07:00,24.3,25.0


In [15]:
# Mark rows where PM2.5 exceeds PM10
df_aqi_local['flag_pm25_exceeds_pm10'] = df_aqi_local["pm2_5"] > df_aqi_local["pm10"]
print(f"Flagged {df_aqi_local['flag_pm25_exceeds_pm10'].sum()} rows")

Flagged 4 rows


In [16]:
# Preserve the flagged (uncorrected) dataframe for reporting/exports with flags
df_aqi_with_flags_source = df_aqi_local.copy()
print("Snapshot taken for hourly/daily with-flags outputs before PM2.5 capping")

Snapshot taken for hourly/daily with-flags outputs before PM2.5 capping


In [17]:
# Fix rows where PM2.5 exceeds PM10 using median PM2.5/PM10 ratio (physically consistent adjustment)
pm25_exceeds_mask = df_aqi_local['flag_pm25_exceeds_pm10']
rows_fixed = int(pm25_exceeds_mask.sum())

# Median ratio from valid rows (pm2.5 <= pm10 and pm10 > 0)
valid_ratio = df_aqi_local.loc[(df_aqi_local['pm10'] > 0) & (df_aqi_local['pm2_5'] <= df_aqi_local['pm10'])]
if len(valid_ratio) == 0:
    ratio_med = 1.0
    print("No valid rows for ratio; defaulting ratio_med=1.0")
else:
    ratio_med = (valid_ratio['pm2_5'] / valid_ratio['pm10']).median()
    print(f"ratio_med (pm2_5/pm10) = {ratio_med:.2f}")

if rows_fixed:
    pm25_scaled = ratio_med * df_aqi_local.loc[pm25_exceeds_mask, 'pm10']
    pm25_scaled = pm25_scaled.clip(upper=df_aqi_local.loc[pm25_exceeds_mask, 'pm10'])
    df_aqi_local.loc[pm25_exceeds_mask, 'pm2_5'] = pm25_scaled.round(2)
    # Recompute the flag after correction to reflect the clean data
    df_aqi_local['flag_pm25_exceeds_pm10'] = df_aqi_local['pm2_5'] > df_aqi_local['pm10']
    print(f"Corrected {rows_fixed} rows where PM2.5 exceeded PM10")
else:
    print("No PM2.5 > PM10 rows to correct")

ratio_med (pm2_5/pm10) = 0.68
Corrected 4 rows where PM2.5 exceeded PM10


In [18]:
# Aftr fix
df_aqi_local[pm_invalid][["pm10", "pm2_5"]].head(10)

,pm10,pm2_5
date,,
2023-12-14 07:00:00+07:00,36.0,24.35
2023-12-14 08:00:00+07:00,33.8,22.86
2023-12-14 09:00:00+07:00,27.5,18.60
2023-12-14 10:00:00+07:00,24.3,16.43


In [19]:
# Check for negative UV index values
uv_cols = ["uv_index", "uv_index_clear_sky"]
uv_negative = (df_aqi_local[uv_cols] < 0).any(axis=1)
print(f"Found {uv_negative.sum()} rows with negative UV index")
df_aqi_local[uv_negative][uv_cols].head(10)

Found 0 rows with negative UV index


,uv_index,uv_index_clear_sky
date,,


In [20]:
# Add a flag for any negative UV index
df_aqi_local['flag_uv_negative'] = (df_aqi_local[uv_cols] < 0).any(axis=1)
print(f"Flagged {df_aqi_local['flag_uv_negative'].sum()} rows with negative UV")

Flagged 0 rows with negative UV


In [21]:
# Ensure UV index stays within the physical range [0, 15]
uv_out_of_range = ((df_aqi_local[uv_cols] < 0) | (df_aqi_local[uv_cols] > 15)).any(axis=1)
df_aqi_local['flag_uv_out_of_range'] = uv_out_of_range
print(f"Flagged {int(df_aqi_local['flag_uv_out_of_range'].sum())} rows with UV index outside [0, 15]")

Flagged 0 rows with UV index outside [0, 15]


In [22]:
# Check PM10 range [0, 1000] µg/m³
pm10_out_of_range = (df_aqi_local["pm10"] < 0) | (df_aqi_local["pm10"] > 1000)
df_aqi_local['flag_pm10_out_of_range'] = pm10_out_of_range
print(f"Flagged {int(df_aqi_local['flag_pm10_out_of_range'].sum())} rows with PM10 outside [0, 1000] µg/m³")

Flagged 0 rows with PM10 outside [0, 1000] µg/m³


In [23]:
# Check PM2.5 range [0, 500] µg/m³
pm25_out_of_range = (df_aqi_local["pm2_5"] < 0) | (df_aqi_local["pm2_5"] > 500)
df_aqi_local['flag_pm25_out_of_range'] = pm25_out_of_range
print(f"Flagged {int(df_aqi_local['flag_pm25_out_of_range'].sum())} rows with PM2.5 outside [0, 500] µg/m³")

Flagged 0 rows with PM2.5 outside [0, 500] µg/m³


In [24]:
# Check if any timestamps are duplicated
duplicates = df_aqi_local.index.duplicated()
print(f"Found {duplicates.sum()} duplicate timestamps")
df_aqi_local[duplicates].head()

Found 0 duplicate timestamps


,pm10,pm2_5,uv_index,uv_index_clear_sky,nitrogen_dioxide,sulphur_dioxide,ozone,carbon_monoxide,flag_missing_pollutants,flag_negative_values,flag_pm25_exceeds_pm10,flag_uv_negative,flag_uv_out_of_range,flag_pm10_out_of_range,flag_pm25_out_of_range
date,,,,,,,,,,,,,,,


In [25]:
# Flag all duplicate timestamps (keep only the first one)
df_aqi_local['flag_duplicate_timestamp'] = df_aqi_local.index.duplicated(keep='first')
print(f"Flagged {df_aqi_local['flag_duplicate_timestamp'].sum()} duplicate timestamps")

Flagged 0 duplicate timestamps


- Time gaps larger than 1 hour

In [26]:
# Check gaps between consecutive timestamps
time_diff = df_aqi_local.index.to_series().diff()
hour_gaps = time_diff > pd.Timedelta(hours=1)
print(f"Found {hour_gaps.sum()} gaps larger than 1 hour")
print(f"Max gap: {time_diff.max()}")

Found 0 gaps larger than 1 hour
Max gap: 0 days 01:00:00


In [27]:
df_aqi_local['time_gap'] = time_diff
large_gaps = df_aqi_local[hour_gaps][['time_gap']].head(10)
large_gaps# Inspect a few of the largest gaps


,time_gap
date,


In [28]:
# Add a flag for rows that sit on a large time gap
df_aqi_local['flag_hour_gap'] = hour_gaps.fillna(False)
print(f"Flagged {df_aqi_local['flag_hour_gap'].sum()} hour gaps")
df_aqi_local = df_aqi_local.drop(columns=['time_gap'])  # drop helper column

Flagged 0 hour gaps


In [29]:
# Collect all QA flag columns in one place
flag_cols = [
    'flag_missing_pollutants',
    'flag_negative_values', 
    'flag_pm25_exceeds_pm10',
    'flag_uv_negative',
    'flag_uv_out_of_range',
    'flag_pm25_out_of_range',
    'flag_pm10_out_of_range',
    'flag_duplicate_timestamp',
    'flag_hour_gap',
]

# Ensure snapshot has all expected flag columns
for col in flag_cols:
    if col not in df_aqi_with_flags_source.columns:
        df_aqi_with_flags_source[col] = False

# Use the uncorrected snapshot for QA summaries/with-flags outputs
flag_summary = df_aqi_with_flags_source[flag_cols].sum()
print("QA Flags Summary (uncorrected snapshot):")
print(flag_summary)
print(f"\nTotal rows with any flag: {df_aqi_with_flags_source[flag_cols].any(axis=1).sum()}")

QA Flags Summary (uncorrected snapshot):
flag_missing_pollutants     0
flag_negative_values        0
flag_pm25_exceeds_pm10      4
flag_uv_negative            0
flag_uv_out_of_range        0
flag_pm25_out_of_range      0
flag_pm10_out_of_range      0
flag_duplicate_timestamp    0
flag_hour_gap               0
dtype: int64

Total rows with any flag: 4


- Check rows that have any QA flag

In [30]:
# Keep only the rows that triggered at least one QA flag (using uncorrected snapshot)
df_aqi_with_flags_source['total_flags'] = df_aqi_with_flags_source[flag_cols].sum(axis=1)
df_flagged_rows = df_aqi_with_flags_source[df_aqi_with_flags_source['total_flags'] > 0]
print(f"Total flagged rows: {len(df_flagged_rows)}")
df_flagged_rows.head()

Total flagged rows: 4


,pm10,pm2_5,uv_index,uv_index_clear_sky,nitrogen_dioxide,sulphur_dioxide,ozone,carbon_monoxide,flag_missing_pollutants,flag_negative_values,flag_pm25_exceeds_pm10,flag_uv_negative,flag_uv_out_of_range,flag_pm25_out_of_range,flag_pm10_out_of_range,flag_duplicate_timestamp,flag_hour_gap,total_flags
date,,,,,,,,,,,,,,,,,,
2023-12-14 07:00:00+07:00,36.0,37.7,0.50,0.50,24.6,22.6,27.0,674.0,False,False,True,False,False,False,False,False,False,1
2023-12-14 08:00:00+07:00,33.8,39.5,2.10,2.10,21.0,22.3,44.0,560.0,False,False,True,False,False,False,False,False,False,1
2023-12-14 09:00:00+07:00,27.5,32.0,4.65,4.65,15.9,21.7,68.0,402.0,False,False,True,False,False,False,False,False,False,1
2023-12-14 10:00:00+07:00,24.3,25.0,7.40,7.45,11.7,20.4,85.0,277.0,False,False,True,False,False,False,False,False,False,1


- Build QA summary report for AQI

In [31]:
# Build a compact QA summary table for air-quality data (using uncorrected snapshot)
aqi_qa_summary = pd.DataFrame({
    'rule': flag_cols,
    'num_flagged': [df_aqi_with_flags_source[col].sum() for col in flag_cols],
    'total_rows': len(df_aqi_with_flags_source),
})

aqi_qa_summary['percent'] = (aqi_qa_summary['num_flagged'] / aqi_qa_summary['total_rows'] * 100).round(2)

# Human-readable descriptions and suggested actions for each rule
rule_descriptions = {
    'flag_missing_pollutants': 'Missing key pollutant values',
    'flag_negative_values': 'Negative readings that are not physically valid',
    'flag_pm25_exceeds_pm10': 'PM2.5 measured higher than PM10',
    'flag_uv_negative': 'Negative UV index values',
    'flag_uv_out_of_range': 'UV index outside the physical 0-15 range',
    'flag_pm25_out_of_range': 'PM2.5 outside the physical 0-500 µg/m³ range',
    'flag_pm10_out_of_range': 'PM10 outside the physical 0-1000 µg/m³ range',
    'flag_duplicate_timestamp': 'Duplicate timestamps in the time series',
    'flag_hour_gap': 'Time gap larger than one hour between rows',
}

rule_actions = {
    'flag_missing_pollutants': 'Keep these rows flagged and review manually',
    'flag_negative_values': 'Replace by 0 or interpolate around them',
    'flag_pm25_exceeds_pm10': 'Auto-cap PM2.5 by median (pm2.5/pm10); keep flag for traceability',
    'flag_uv_negative': 'Replace negative values by 0',
    'flag_uv_out_of_range': 'Investigate instrument issues; clip to [0, 15] if confirmed valid',
    'flag_pm25_out_of_range': 'Investigate sensor calibration; clip to [0, 500] if confirmed valid',
    'flag_pm10_out_of_range': 'Investigate sensor calibration; clip to [0, 1000] if confirmed valid',
    'flag_duplicate_timestamp': 'Keep the first record for each timestamp',
    'flag_hour_gap': 'Keep flagged and mention gaps in the analysis notes',
}

aqi_qa_summary['description'] = aqi_qa_summary['rule'].map(rule_descriptions)
aqi_qa_summary['action'] = aqi_qa_summary['rule'].map(rule_actions)
aqi_qa_summary['dataset'] = 'air_quality'
aqi_qa_summary['level'] = 'hourly'

print("AQI QA Summary (uncorrected snapshot):")
print(aqi_qa_summary)
aqi_qa_summary

AQI QA Summary (uncorrected snapshot):
                       rule  num_flagged  total_rows  percent  \
0   flag_missing_pollutants            0        8760     0.00   
1      flag_negative_values            0        8760     0.00   
2    flag_pm25_exceeds_pm10            4        8760     0.05   
3          flag_uv_negative            0        8760     0.00   
4      flag_uv_out_of_range            0        8760     0.00   
5    flag_pm25_out_of_range            0        8760     0.00   
6    flag_pm10_out_of_range            0        8760     0.00   
7  flag_duplicate_timestamp            0        8760     0.00   
8             flag_hour_gap            0        8760     0.00   

                                       description  \
0                     Missing key pollutant values   
1  Negative readings that are not physically valid   
2                  PM2.5 measured higher than PM10   
3                         Negative UV index values   
4         UV index outside the physical 

,rule,num_flagged,total_rows,percent,description,action,dataset,level
0,flag_missing_pollutants,0,8760,0.00,Missing key pollutant values,Keep these rows flagged and review manually,air_quality,hourly
1,flag_negative_values,0,8760,0.00,Negative readings that are not physically valid,Replace by 0 or interpolate around them,air_quality,hourly
2,flag_pm25_exceeds_pm10,4,8760,0.05,PM2.5 measured higher than PM10,Auto-cap PM2.5 by median (pm2.5/pm10); keep fl...,air_quality,hourly
3,flag_uv_negative,0,8760,0.00,Negative UV index values,Replace negative values by 0,air_quality,hourly
4,flag_uv_out_of_range,0,8760,0.00,UV index outside the physical 0-15 range,"Investigate instrument issues; clip to [0, 15]...",air_quality,hourly
5,flag_pm25_out_of_range,0,8760,0.00,PM2.5 outside the physical 0-500 µg/m³ range,"Investigate sensor calibration; clip to [0, 50...",air_quality,hourly
6,flag_pm10_out_of_range,0,8760,0.00,PM10 outside the physical 0-1000 µg/m³ range,"Investigate sensor calibration; clip to [0, 10...",air_quality,hourly
7,flag_duplicate_timestamp,0,8760,0.00,Duplicate timestamps in the time series,Keep the first record for each timestamp,air_quality,hourly
8,flag_hour_gap,0,8760,0.00,Time gap larger than one hour between rows,Keep flagged and mention gaps in the analysis ...,air_quality,hourly


In [32]:
# Save the QA summary to a CSV file for later reference
os.makedirs(REPORTS_DIR, exist_ok=True)
qa_aqi_path = os.path.join(REPORTS_DIR, "qa_aqi_hourly.csv")
aqi_qa_summary.to_csv(qa_aqi_path, index=False)
print(f"Saved AQI QA report to {qa_aqi_path}")

Saved AQI QA report to ../reports\qa_aqi_hourly.csv


In [33]:
# Export HOURLY AQI with flags (traceable snapshot before daily resample)
os.makedirs(PROCESSED_DIR, exist_ok=True)

aqi_flags_path = os.path.join(PROCESSED_DIR, "hourly_aqi_with_flags.csv")
df_aqi_flags = df_aqi_with_flags_source.copy()

num_cols = df_aqi_flags.select_dtypes(include="number").columns
df_aqi_flags[num_cols] = df_aqi_flags[num_cols].round(2)

df_aqi_flags.reset_index().to_csv(aqi_flags_path, index=False)
print(f"Saved hourly AQI with flags to {aqi_flags_path}")
df_aqi_flags.head()

Saved hourly AQI with flags to ../processed\hourly_aqi_with_flags.csv


,pm10,pm2_5,uv_index,uv_index_clear_sky,nitrogen_dioxide,sulphur_dioxide,ozone,carbon_monoxide,flag_missing_pollutants,flag_negative_values,flag_pm25_exceeds_pm10,flag_uv_negative,flag_uv_out_of_range,flag_pm25_out_of_range,flag_pm10_out_of_range,flag_duplicate_timestamp,flag_hour_gap,total_flags
date,,,,,,,,,,,,,,,,,,
2023-01-01 00:00:00+07:00,95.0,65.0,0.0,0.0,84.40,37.50,25.0,993.0,False,False,False,False,False,False,False,False,False,0
2023-01-01 01:00:00+07:00,86.1,59.0,0.0,0.0,70.25,30.75,28.0,846.0,False,False,False,False,False,False,False,False,False,0
2023-01-01 02:00:00+07:00,83.4,57.0,0.0,0.0,65.80,28.90,26.0,821.0,False,False,False,False,False,False,False,False,False,0
2023-01-01 03:00:00+07:00,80.1,54.8,0.0,0.0,63.95,28.45,22.0,834.0,False,False,False,False,False,False,False,False,False,0
2023-01-01 04:00:00+07:00,69.4,47.5,0.0,0.0,60.10,27.20,20.0,838.0,False,False,False,False,False,False,False,False,False,0


- Convert to daily level after cleaning

In [34]:
df_aqi_local.head()

,pm10,pm2_5,uv_index,uv_index_clear_sky,nitrogen_dioxide,sulphur_dioxide,ozone,carbon_monoxide,flag_missing_pollutants,flag_negative_values,flag_pm25_exceeds_pm10,flag_uv_negative,flag_uv_out_of_range,flag_pm10_out_of_range,flag_pm25_out_of_range,flag_duplicate_timestamp,flag_hour_gap
date,,,,,,,,,,,,,,,,,
2023-01-01 00:00:00+07:00,95.0,65.0,0.0,0.0,84.40,37.50,25.0,993.0,False,False,False,False,False,False,False,False,False
2023-01-01 01:00:00+07:00,86.1,59.0,0.0,0.0,70.25,30.75,28.0,846.0,False,False,False,False,False,False,False,False,False
2023-01-01 02:00:00+07:00,83.4,57.0,0.0,0.0,65.80,28.90,26.0,821.0,False,False,False,False,False,False,False,False,False
2023-01-01 03:00:00+07:00,80.1,54.8,0.0,0.0,63.95,28.45,22.0,834.0,False,False,False,False,False,False,False,False,False
2023-01-01 04:00:00+07:00,69.4,47.5,0.0,0.0,60.10,27.20,20.0,838.0,False,False,False,False,False,False,False,False,False


In [35]:
# check clean
df_aqi_local['total_flags'] = df_aqi_local[flag_cols].sum(axis=1)
df_flagged_rows = df_aqi_local[df_aqi_local['total_flags'] > 0]
print(f"Total flagged rows: {len(df_flagged_rows)}")
df_flagged_rows.head()

Total flagged rows: 0


,pm10,pm2_5,uv_index,uv_index_clear_sky,nitrogen_dioxide,sulphur_dioxide,ozone,carbon_monoxide,flag_missing_pollutants,flag_negative_values,flag_pm25_exceeds_pm10,flag_uv_negative,flag_uv_out_of_range,flag_pm10_out_of_range,flag_pm25_out_of_range,flag_duplicate_timestamp,flag_hour_gap,total_flags
date,,,,,,,,,,,,,,,,,,


In [36]:
# Create a clean version for resampling (drop QA flags and helper count)
drop_cols = [c for c in flag_cols + ['total_flags'] if c in df_aqi_local.columns]
df_aqi_clean = df_aqi_local.drop(columns=drop_cols)
print(f"Clean dataset shape: {df_aqi_clean.shape}")
print(f"Year {YEAR} rows: {len(df_aqi_clean[df_aqi_clean.index.year == YEAR])}")
df_aqi_clean.head(10)

Clean dataset shape: (8760, 8)
Year 2023 rows: 8760


,pm10,pm2_5,uv_index,uv_index_clear_sky,nitrogen_dioxide,sulphur_dioxide,ozone,carbon_monoxide
date,,,,,,,,
2023-01-01 00:00:00+07:00,95.0,65.0,0.00,0.00,84.40,37.50,25.0,993.0
2023-01-01 01:00:00+07:00,86.1,59.0,0.00,0.00,70.25,30.75,28.0,846.0
2023-01-01 02:00:00+07:00,83.4,57.0,0.00,0.00,65.80,28.90,26.0,821.0
2023-01-01 03:00:00+07:00,80.1,54.8,0.00,0.00,63.95,28.45,22.0,834.0
2023-01-01 04:00:00+07:00,69.4,47.5,0.00,0.00,60.10,27.20,20.0,838.0
2023-01-01 05:00:00+07:00,57.9,39.5,0.00,0.00,53.70,24.50,22.0,805.0
2023-01-01 06:00:00+07:00,48.8,33.2,0.00,0.00,45.95,21.05,26.0,759.0
2023-01-01 07:00:00+07:00,31.5,21.5,0.15,0.35,23.30,6.65,48.0,555.0
2023-01-01 08:00:00+07:00,29.9,20.3,0.85,1.55,19.30,6.45,56.0,529.0


In [37]:
# Resample from hourly to daily values
aggregation_rules = {
    'pm10': 'mean',
    'pm2_5': 'mean',
    'nitrogen_dioxide': 'mean',  
    'sulphur_dioxide': 'mean',  
    'ozone': 'mean',
    'carbon_monoxide': 'mean',
    'uv_index': 'max',  # take the daily maximum UV index
    'uv_index_clear_sky': 'max', 
}

df_aqi_daily = df_aqi_clean.resample('D').agg(aggregation_rules).round(2)

# Rename UV columns to make their meaning clear
df_aqi_daily = df_aqi_daily.rename(columns={
    'uv_index': 'uv_index_max',
    'uv_index_clear_sky': 'uv_index_clear_sky_max',
})
df_aqi_daily

,pm10,pm2_5,nitrogen_dioxide,sulphur_dioxide,ozone,carbon_monoxide,uv_index_max,uv_index_clear_sky_max
date,,,,,,,,
2023-01-01 00:00:00+07:00,42.38,29.00,42.76,17.00,51.33,710.33,7.20,8.80
2023-01-02 00:00:00+07:00,51.38,35.54,53.81,9.94,57.96,1090.38,7.55,8.30
2023-01-03 00:00:00+07:00,40.10,27.82,39.09,5.94,54.12,662.62,7.65,8.45
2023-01-04 00:00:00+07:00,28.11,19.46,28.41,13.16,46.79,495.75,6.30,9.10
2023-01-05 00:00:00+07:00,16.40,11.28,21.09,11.22,51.29,352.62,3.15,9.55
...,...,...,...,...,...,...,...,...
2023-12-27 00:00:00+07:00,60.06,40.11,33.31,38.73,82.08,749.21,8.65,8.65
2023-12-28 00:00:00+07:00,37.68,25.13,25.58,23.48,53.62,544.08,9.20,9.45
2023-12-29 00:00:00+07:00,39.24,26.45,30.91,32.62,56.42,519.17,8.70,9.25


In [38]:
# Save the daily AQI dataset for later steps
df_aqi_daily.to_csv("../processed/daily_aqi_processed.csv")
df_aqi_daily.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 365 entries, 2023-01-01 00:00:00+07:00 to 2023-12-31 00:00:00+07:00
Freq: D
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   pm10                    365 non-null    float64
 1   pm2_5                   365 non-null    float64
 2   nitrogen_dioxide        365 non-null    float64
 3   sulphur_dioxide         365 non-null    float64
 4   ozone                   365 non-null    float64
 5   carbon_monoxide         365 non-null    float64
 6   uv_index_max            365 non-null    float64
 7   uv_index_clear_sky_max  365 non-null    float64
dtypes: float64(8)
memory usage: 25.7 KB


In [39]:
# Export DAILY AQI dataset WITH FLAGS (parallel to weather output)
os.makedirs(PROCESSED_DIR, exist_ok=True)

agg_rules_flags = {
    "pm10": "mean",
    "pm2_5": "mean",
    "nitrogen_dioxide": "mean",
    "sulphur_dioxide": "mean",
    "ozone": "mean",
    "carbon_monoxide": "mean",
    # keep UV as daily maxima like the clean output
    "uv_index": "max",
    "uv_index_clear_sky": "max",
}

# Flags -> any; total_flags -> sum
for col in flag_cols:
    agg_rules_flags[col] = "any"
if "total_flags" in df_aqi_with_flags_source.columns:
    agg_rules_flags["total_flags"] = "sum"

# Fallback for any remaining columns
for col in df_aqi_with_flags_source.columns:
    if col not in agg_rules_flags:
        agg_rules_flags[col] = "first"

# Resample to daily with flags preserved (using uncorrected snapshot)
df_aqi_daily_with_flags = df_aqi_with_flags_source.resample("D").agg(agg_rules_flags)

# Rename UV columns for clarity (max values)
df_aqi_daily_with_flags = df_aqi_daily_with_flags.rename(columns={
    "uv_index": "uv_index_max",
    "uv_index_clear_sky": "uv_index_clear_sky_max",
})

# Round numeric columns to 2 decimal places for output clarity
numeric_cols = df_aqi_daily_with_flags.select_dtypes(include="number").columns
df_aqi_daily_with_flags[numeric_cols] = df_aqi_daily_with_flags[numeric_cols].round(2)

# Write to file with an explicit date column
aqi_daily_with_flags_path = os.path.join(PROCESSED_DIR, "daily_aqi_with_flags.csv")
df_aqi_daily_with_flags.reset_index().to_csv(aqi_daily_with_flags_path, index=False)
print(f"Saved daily AQI with flags to {aqi_daily_with_flags_path}")
df_aqi_daily_with_flags.head()

Saved daily AQI with flags to ../processed\daily_aqi_with_flags.csv


,pm10,pm2_5,nitrogen_dioxide,sulphur_dioxide,ozone,carbon_monoxide,uv_index_max,uv_index_clear_sky_max,flag_missing_pollutants,flag_negative_values,flag_pm25_exceeds_pm10,flag_uv_negative,flag_uv_out_of_range,flag_pm25_out_of_range,flag_pm10_out_of_range,flag_duplicate_timestamp,flag_hour_gap,total_flags
date,,,,,,,,,,,,,,,,,,
2023-01-01 00:00:00+07:00,42.38,29.00,42.76,17.00,51.33,710.33,7.20,8.80,False,False,False,False,False,False,False,False,False,0
2023-01-02 00:00:00+07:00,51.38,35.54,53.81,9.94,57.96,1090.38,7.55,8.30,False,False,False,False,False,False,False,False,False,0
2023-01-03 00:00:00+07:00,40.10,27.82,39.09,5.94,54.12,662.62,7.65,8.45,False,False,False,False,False,False,False,False,False,0
2023-01-04 00:00:00+07:00,28.11,19.46,28.41,13.16,46.79,495.75,6.30,9.10,False,False,False,False,False,False,False,False,False,0
2023-01-05 00:00:00+07:00,16.40,11.28,21.09,11.22,51.29,352.62,3.15,9.55,False,False,False,False,False,False,False,False,False,0


# Weather data
* Ensure transparent data quality assurance (QA)
* Prioritize flagging over deleting data
* Synchronize timestamps across all data sources
* Standardize measurement units and formats
* Resample and unify data before merging sources (bởi vì chỉ có dữ liệu theo ngày và tần suất như nhau nên sẽ skip phần này)
* Log and report all QA results

In [40]:
# Read weather data and turn blank values into NaN
df_weather = pd.read_csv(
    RAW_WEATHER_FILE,
    na_values=["", " ", "-", "NULL"],
    keep_default_na=True, 
    skipinitialspace=True,
)
df_weather = df_weather.replace(r"^\s*$", np.nan, regex=True)

In [41]:
df_weather.head(10)

,time,tavg,tmin,tmax,prcp,snow,wdir,wspd,wpgt,pres,tsun
0,2023-01-01,26.5,23.0,31.0,0.6,NaN,NaN,8.9,NaN,1012.7,NaN
1,2023-01-02,26.7,23.0,32.0,0.0,NaN,NaN,6.3,NaN,1013.2,NaN
2,2023-01-03,28.1,24.0,33.0,1.0,NaN,NaN,9.7,NaN,1012.4,NaN
3,2023-01-04,27.6,24.0,33.0,0.1,NaN,NaN,13.4,NaN,1011.7,NaN
4,2023-01-05,25.8,24.0,28.0,5.8,NaN,NaN,17.2,NaN,1011.3,NaN
5,2023-01-06,25.7,22.0,30.0,18.3,NaN,NaN,15.2,NaN,1010.9,NaN
6,2023-01-07,25.9,24.0,29.0,1.9,NaN,NaN,13.6,NaN,1010.3,NaN
7,2023-01-08,25.9,23.0,29.0,4.3,NaN,NaN,6.6,NaN,1010.7,NaN
8,2023-01-09,26.8,25.0,31.0,23.6,NaN,NaN,7.7,NaN,1011.6,NaN
9,2023-01-10,26.3,24.0,29.0,16.2,NaN,NaN,12.2,NaN,1011.2,NaN


In [42]:
# Normalize numeric columns and drop all-NaN columns for cleaner QA (similar to the AQI normalization step)
weather_numeric_cols = [
    "tavg", "tmin", "tmax",
    "prcp", "snow",
    "wdir", "wspd", "wpgt",
    "pres", "tsun",
]

for col in weather_numeric_cols:
    if col in df_weather.columns:
        df_weather[col] = pd.to_numeric(df_weather[col], errors="coerce")

# Drop columns that are all NaN (except the time column)
all_nan_cols = [
    c for c in df_weather.columns
    if c != "time" and df_weather[c].isna().all()
]
if all_nan_cols:
    print("Dropping all-NaN weather columns:", all_nan_cols)
    df_weather = df_weather.drop(columns=all_nan_cols)

weather_numeric_cols = [c for c in weather_numeric_cols if c in df_weather.columns]
print("Numeric columns kept:", weather_numeric_cols)
df_weather.head()


Dropping all-NaN weather columns: ['snow', 'wdir', 'wpgt', 'tsun']
Numeric columns kept: ['tavg', 'tmin', 'tmax', 'prcp', 'wspd', 'pres']


,time,tavg,tmin,tmax,prcp,wspd,pres
0,2023-01-01,26.5,23.0,31.0,0.6,8.9,1012.7
1,2023-01-02,26.7,23.0,32.0,0.0,6.3,1013.2
2,2023-01-03,28.1,24.0,33.0,1.0,9.7,1012.4
3,2023-01-04,27.6,24.0,33.0,0.1,13.4,1011.7
4,2023-01-05,25.8,24.0,28.0,5.8,17.2,1011.3


In [43]:
# Normalize the time column and build an index (similar to df_aqi_local)
ts = pd.to_datetime(df_weather["time"], errors="coerce")

bad_time = ts.isna().sum()
if bad_time:
    print(f"Dropping {bad_time} rows with invalid time")

df_weather = df_weather.loc[~ts.isna()].copy()

ts = ts.loc[~ts.isna()]
if getattr(ts.dtype, "tz", None) is None:
    ts = ts.dt.tz_localize(TIMEZONE)
else:
    ts = ts.dt.tz_convert(TIMEZONE)

df_weather["time"] = ts

df_weather_local = (
    df_weather
    .set_index("time")
    .sort_index()
)

print("Weather time range:", df_weather_local.index.min(), "→", df_weather_local.index.max())
print("Weather raw shape:", df_weather_local.shape)


Weather time range: 2023-01-01 00:00:00+07:00 → 2023-12-31 00:00:00+07:00
Weather raw shape: (365, 6)


In [44]:
# Check missing temperature fields (tavg/tmin/tmax)
missing_temps = df_weather_local[[c for c in ["tavg", "tmin", "tmax"] if c in df_weather_local.columns]].isna().any(axis=1)
print("Rows with missing temperature fields:", int(missing_temps.sum()))
missing_temps.head()

Rows with missing temperature fields: 0


time
2023-01-01 00:00:00+07:00    False
2023-01-02 00:00:00+07:00    False
2023-01-03 00:00:00+07:00    False
2023-01-04 00:00:00+07:00    False
2023-01-05 00:00:00+07:00    False
dtype: bool

In [45]:
# Flag missing temperature fields
if not {"tavg", "tmin", "tmax"}.issubset(df_weather_local.columns):
    df_weather_local["flag_missing_temps"] = False
else:
    df_weather_local["flag_missing_temps"] = missing_temps
print(f"Flagged {int(df_weather_local['flag_missing_temps'].sum())} rows with missing temps")

Flagged 0 rows with missing temps


In [46]:
# Check missing precipitation
a_prcp = "prcp" in df_weather_local.columns
missing_precip = df_weather_local["prcp"].isna() if a_prcp else pd.Series(False, index=df_weather_local.index)
print("Rows missing precipitation:", int(missing_precip.sum()))
missing_precip.head()

Rows missing precipitation: 0


time
2023-01-01 00:00:00+07:00    False
2023-01-02 00:00:00+07:00    False
2023-01-03 00:00:00+07:00    False
2023-01-04 00:00:00+07:00    False
2023-01-05 00:00:00+07:00    False
Name: prcp, dtype: bool

In [47]:
# Flag missing precipitation
df_weather_local["flag_missing_precip"] = missing_precip
print(f"Flagged {int(df_weather_local['flag_missing_precip'].sum())} rows missing precipitation")

Flagged 0 rows missing precipitation


In [48]:
# Check missing wind speed (wspd)
wspd_present = "wspd" in df_weather_local.columns
missing_wind = df_weather_local["wspd"].isna() if wspd_present else pd.Series(False, index=df_weather_local.index)
print("Rows missing wind speed:", int(missing_wind.sum()))
missing_wind.head()

Rows missing wind speed: 0


time
2023-01-01 00:00:00+07:00    False
2023-01-02 00:00:00+07:00    False
2023-01-03 00:00:00+07:00    False
2023-01-04 00:00:00+07:00    False
2023-01-05 00:00:00+07:00    False
Name: wspd, dtype: bool

In [49]:
# Flag missing wind speed
df_weather_local["flag_missing_wind"] = missing_wind
print(f"Flagged {int(df_weather_local['flag_missing_wind'].sum())} rows missing wind speed")

Flagged 0 rows missing wind speed


In [50]:
# Check missing pressure
missing_pressure = df_weather_local["pres"].isna() if "pres" in df_weather_local.columns else pd.Series(False, index=df_weather_local.index)
print("Rows missing pressure:", int(missing_pressure.sum()))
missing_pressure.head()

Rows missing pressure: 0


time
2023-01-01 00:00:00+07:00    False
2023-01-02 00:00:00+07:00    False
2023-01-03 00:00:00+07:00    False
2023-01-04 00:00:00+07:00    False
2023-01-05 00:00:00+07:00    False
Name: pres, dtype: bool

In [51]:
# Flag missing pressure
df_weather_local["flag_missing_pressure"] = missing_pressure
print(f"Flagged {int(df_weather_local['flag_missing_pressure'].sum())} rows missing pressure")

Flagged 0 rows missing pressure


In [52]:
# Check tmin > tmax (invalid relation)
if {"tmin", "tmax"}.issubset(df_weather_local.columns):
    invalid_tmin_tmax = df_weather_local["tmin"] > df_weather_local["tmax"]
else:
    invalid_tmin_tmax = pd.Series(False, index=df_weather_local.index)
print("Rows where tmin > tmax:", int(invalid_tmin_tmax.sum()))
invalid_tmin_tmax.head()

Rows where tmin > tmax: 0


time
2023-01-01 00:00:00+07:00    False
2023-01-02 00:00:00+07:00    False
2023-01-03 00:00:00+07:00    False
2023-01-04 00:00:00+07:00    False
2023-01-05 00:00:00+07:00    False
dtype: bool

In [53]:
# Flag tmin > tmax
df_weather_local["flag_invalid_tmin_tmax_relation"] = invalid_tmin_tmax
print(f"Flagged {int(df_weather_local['flag_invalid_tmin_tmax_relation'].sum())} rows with tmin > tmax")

Flagged 0 rows with tmin > tmax


In [54]:
# Check tavg outside [tmin, tmax]
if {"tavg", "tmin", "tmax"}.issubset(df_weather_local.columns):
    invalid_tavg_range = (df_weather_local["tavg"] < df_weather_local["tmin"]) | (df_weather_local["tavg"] > df_weather_local["tmax"])
else:
    invalid_tavg_range = pd.Series(False, index=df_weather_local.index)
print("Rows where tavg outside [tmin, tmax]:", int(invalid_tavg_range.sum()))
invalid_tavg_range.head()

Rows where tavg outside [tmin, tmax]: 0


time
2023-01-01 00:00:00+07:00    False
2023-01-02 00:00:00+07:00    False
2023-01-03 00:00:00+07:00    False
2023-01-04 00:00:00+07:00    False
2023-01-05 00:00:00+07:00    False
dtype: bool

In [55]:
# Flag tavg outside [tmin, tmax]
df_weather_local["flag_invalid_tavg_out_of_range"] = invalid_tavg_range
print(f"Flagged {int(df_weather_local['flag_invalid_tavg_out_of_range'].sum())} rows with tavg outside range")

Flagged 0 rows with tavg outside range


In [56]:
# Check extreme temperature range
if {"tmin", "tmax"}.issubset(df_weather_local.columns):
    extreme_temp = (df_weather_local["tmax"] > 60) | (df_weather_local["tmin"] < -80)
else:
    extreme_temp = pd.Series(False, index=df_weather_local.index)
print("Rows with extreme temperature:", int(extreme_temp.sum()))
extreme_temp.head()

Rows with extreme temperature: 0


time
2023-01-01 00:00:00+07:00    False
2023-01-02 00:00:00+07:00    False
2023-01-03 00:00:00+07:00    False
2023-01-04 00:00:00+07:00    False
2023-01-05 00:00:00+07:00    False
dtype: bool

In [57]:
# Flag extreme temperature
df_weather_local["flag_extreme_temperature"] = extreme_temp
print(f"Flagged {int(df_weather_local['flag_extreme_temperature'].sum())} rows with extreme temperature")

Flagged 0 rows with extreme temperature


In [58]:
# Check negative precipitation
if "prcp" in df_weather_local.columns:
    negative_precip = df_weather_local["prcp"] < 0
else:
    negative_precip = pd.Series(False, index=df_weather_local.index)
print("Rows with negative precipitation:", int(negative_precip.sum()))
negative_precip.head()

Rows with negative precipitation: 0


time
2023-01-01 00:00:00+07:00    False
2023-01-02 00:00:00+07:00    False
2023-01-03 00:00:00+07:00    False
2023-01-04 00:00:00+07:00    False
2023-01-05 00:00:00+07:00    False
Name: prcp, dtype: bool

In [59]:
# Flag negative precipitation
df_weather_local["flag_negative_precipitation"] = negative_precip
print(f"Flagged {int(df_weather_local['flag_negative_precipitation'].sum())} rows with negative precip")

Flagged 0 rows with negative precip


In [60]:
# Check negative wind speed
if "wspd" in df_weather_local.columns:
    negative_wind_speed = df_weather_local["wspd"] < 0
else:
    negative_wind_speed = pd.Series(False, index=df_weather_local.index)
print("Rows with negative wind speed:", int(negative_wind_speed.sum()))
negative_wind_speed.head()

Rows with negative wind speed: 0


time
2023-01-01 00:00:00+07:00    False
2023-01-02 00:00:00+07:00    False
2023-01-03 00:00:00+07:00    False
2023-01-04 00:00:00+07:00    False
2023-01-05 00:00:00+07:00    False
Name: wspd, dtype: bool

In [61]:
# Flag negative wind speed or gust
df_weather_local["flag_negative_wind_speed"] = negative_wind_speed
print(f"Flagged {int(df_weather_local['flag_negative_wind_speed'].sum())} rows with negative wind speed/gust")

Flagged 0 rows with negative wind speed/gust


In [62]:
# Check pressure outside [870, 1100] hPa
if "pres" in df_weather_local.columns:
    pressure_outlier = (df_weather_local["pres"] < 870) | (df_weather_local["pres"] > 1100)
else:
    pressure_outlier = pd.Series(False, index=df_weather_local.index)
print("Rows with pressure outlier:", int(pressure_outlier.sum()))
pressure_outlier.head()

Rows with pressure outlier: 0


time
2023-01-01 00:00:00+07:00    False
2023-01-02 00:00:00+07:00    False
2023-01-03 00:00:00+07:00    False
2023-01-04 00:00:00+07:00    False
2023-01-05 00:00:00+07:00    False
Name: pres, dtype: bool

In [63]:
# Flag pressure outlier
df_weather_local["flag_pressure_outlier"] = pressure_outlier
print(f"Flagged {int(df_weather_local['flag_pressure_outlier'].sum())} rows with pressure outlier")

Flagged 0 rows with pressure outlier


In [64]:
# Check duplicate timestamps
duplicates = df_weather_local.index.duplicated(keep=False)
print("Duplicate timestamps:", int(duplicates.sum()))
df_weather_local[duplicates].head()

Duplicate timestamps: 0


,tavg,tmin,tmax,prcp,wspd,pres,flag_missing_temps,flag_missing_precip,flag_missing_wind,flag_missing_pressure,flag_invalid_tmin_tmax_relation,flag_invalid_tavg_out_of_range,flag_extreme_temperature,flag_negative_precipitation,flag_negative_wind_speed,flag_pressure_outlier
time,,,,,,,,,,,,,,,,


In [65]:
# Flag duplicate timestamps
df_weather_local["flag_duplicate_timestamp"] = duplicates
print(f"Flagged {int(df_weather_local['flag_duplicate_timestamp'].sum())} duplicate timestamps")

Flagged 0 duplicate timestamps


In [66]:
# Check day gaps > 1 day
time_diff = df_weather_local.index.to_series().diff()
day_gaps = time_diff > pd.Timedelta(days=1)
print("Rows sitting on gaps >1 day:", int(day_gaps.sum()))
day_gaps.head()

Rows sitting on gaps >1 day: 0


time
2023-01-01 00:00:00+07:00    False
2023-01-02 00:00:00+07:00    False
2023-01-03 00:00:00+07:00    False
2023-01-04 00:00:00+07:00    False
2023-01-05 00:00:00+07:00    False
Name: time, dtype: bool

In [67]:
# Flag day gaps > 1 day
df_weather_local["flag_day_gap"] = day_gaps.fillna(False)
print(f"Flagged {int(df_weather_local['flag_day_gap'].sum())} rows on large day gaps")

Flagged 0 rows on large day gaps


In [68]:
# Aggregate flag columns and build weather QA statistics
weather_flag_cols = [
    "flag_missing_temps",
    "flag_missing_precip",
    "flag_missing_wind",
    "flag_missing_pressure",
    "flag_invalid_tmin_tmax_relation",
    "flag_invalid_tavg_out_of_range",
    "flag_extreme_temperature",
    "flag_negative_precipitation",
    "flag_negative_wind_speed",
    "flag_pressure_outlier",
    "flag_duplicate_timestamp",
    "flag_day_gap",
]

flag_summary = df_weather_local[weather_flag_cols].sum()
print("Weather QA Flags Summary:")
print(flag_summary)
print(f"\nTotal rows with any flag: {int(df_weather_local[weather_flag_cols].any(axis=1).sum())}")

# Add *_count helper columns to keep the schema consistent with daily_with_flags export
for col in weather_flag_cols:
    df_weather_local[f"{col}_count"] = df_weather_local[col].astype(int)

weather_flag_count_cols = [f"{c}_count" for c in weather_flag_cols]
df_weather_local["weather_any_flag"] = df_weather_local[weather_flag_cols].any(axis=1)
df_weather_local["weather_total_flags_count"] = df_weather_local[weather_flag_count_cols].sum(axis=1)

# Build a QA summary table for weather data
weather_rule_desc = {
    "flag_missing_temps": "Missing temperature values",
    "flag_missing_precip": "Missing precipitation",
    "flag_missing_wind": "Missing wind speed",
    "flag_missing_pressure": "Missing pressure",
    "flag_invalid_tmin_tmax_relation": "Minimum temperature greater than maximum",
    "flag_invalid_tavg_out_of_range": "Average temperature outside min–max range",
    "flag_extreme_temperature": "Temperature outside realistic range (-80, 60)",
    "flag_negative_precipitation": "Negative precipitation is invalid",
    "flag_negative_wind_speed": "Negative wind speed",
    "flag_pressure_outlier": "Pressure outside 870–1100 hPa",
    "flag_duplicate_timestamp": "Duplicate timestamps in the time series",
    "flag_day_gap": "Time gap larger than one day",
}

weather_rule_actions = {k: "Keep flagged; review in analysis/report" for k in weather_rule_desc}

weather_qa_summary = pd.DataFrame({
    "rule": weather_flag_cols,
    "num_flagged": [int(df_weather_local[c].sum()) for c in weather_flag_cols],
    "total_rows": len(df_weather_local),
})
weather_qa_summary["percent"] = (weather_qa_summary["num_flagged"] / weather_qa_summary["total_rows"] * 100).round(2)
weather_qa_summary["description"] = weather_qa_summary["rule"].map(weather_rule_desc)
weather_qa_summary["action"] = weather_qa_summary["rule"].map(weather_rule_actions)
weather_qa_summary["dataset"] = "weather"
weather_qa_summary["level"] = "daily"

weather_qa_summary

Weather QA Flags Summary:
flag_missing_temps                 0
flag_missing_precip                0
flag_missing_wind                  0
flag_missing_pressure              0
flag_invalid_tmin_tmax_relation    0
flag_invalid_tavg_out_of_range     0
flag_extreme_temperature           0
flag_negative_precipitation        0
flag_negative_wind_speed           0
flag_pressure_outlier              0
flag_duplicate_timestamp           0
flag_day_gap                       0
dtype: int64

Total rows with any flag: 0


,rule,num_flagged,total_rows,percent,description,action,dataset,level
0,flag_missing_temps,0,365,0.0,Missing temperature values,Keep flagged; review in analysis/report,weather,daily
1,flag_missing_precip,0,365,0.0,Missing precipitation,Keep flagged; review in analysis/report,weather,daily
2,flag_missing_wind,0,365,0.0,Missing wind speed,Keep flagged; review in analysis/report,weather,daily
3,flag_missing_pressure,0,365,0.0,Missing pressure,Keep flagged; review in analysis/report,weather,daily
4,flag_invalid_tmin_tmax_relation,0,365,0.0,Minimum temperature greater than maximum,Keep flagged; review in analysis/report,weather,daily
5,flag_invalid_tavg_out_of_range,0,365,0.0,Average temperature outside min–max range,Keep flagged; review in analysis/report,weather,daily
6,flag_extreme_temperature,0,365,0.0,"Temperature outside realistic range (-80, 60)",Keep flagged; review in analysis/report,weather,daily
7,flag_negative_precipitation,0,365,0.0,Negative precipitation is invalid,Keep flagged; review in analysis/report,weather,daily
8,flag_negative_wind_speed,0,365,0.0,Negative wind speed,Keep flagged; review in analysis/report,weather,daily
9,flag_pressure_outlier,0,365,0.0,Pressure outside 870–1100 hPa,Keep flagged; review in analysis/report,weather,daily


In [69]:
# Save QA summary for weather and combined dataset (same pattern as AQI)
os.makedirs(REPORTS_DIR, exist_ok=True)
qa_daily_path = os.path.join(REPORTS_DIR, "qa_weather_daily.csv")
weather_qa_summary.to_csv(qa_daily_path, index=False)
print(f"Saved weather QA report to {qa_daily_path}")

combined_qa = pd.concat([aqi_qa_summary, weather_qa_summary], ignore_index=True, sort=False)
combined_qa.to_csv(QA_REPORT_FILE, index=False)
print(f"Saved combined QA report to {QA_REPORT_FILE}")

Saved weather QA report to ../reports\qa_weather_daily.csv
Saved combined QA report to ../reports\qa_summary.csv


In [70]:
# Export DAILY weather datasets (clean + with_flags) without resampling (weather is already daily)
os.makedirs(PROCESSED_DIR, exist_ok=True)

weather_flag_count_cols = [f"{c}_count" for c in weather_flag_cols]

# --- Daily weather WITH FLAGS (transparent/traceable) ---
df_weather_with_flags = df_weather_local.copy().reset_index().rename(columns={"time": "date"})
df_weather_with_flags = df_weather_with_flags.sort_values("date")
numeric_cols = df_weather_with_flags.select_dtypes(include="number").columns
df_weather_with_flags[numeric_cols] = df_weather_with_flags[numeric_cols].round(2)
weather_with_flags_out = os.path.join(PROCESSED_DIR, "daily_weather_with_flags.csv")
df_weather_with_flags.to_csv(weather_with_flags_out, index=False)
print(f"Saved daily weather with flags to {weather_with_flags_out}")

# --- Daily weather CLEAN (no flags) ---
cols_to_drop = weather_flag_cols + weather_flag_count_cols + ["weather_any_flag", "weather_total_flags_count"]
df_weather_clean = df_weather_local.drop(columns=cols_to_drop, errors="ignore")
df_weather_clean_out = df_weather_clean.reset_index().rename(columns={"time": "date"})
df_weather_clean_out = df_weather_clean_out.sort_values("date")
numeric_clean_cols = df_weather_clean_out.select_dtypes(include="number").columns
df_weather_clean_out[numeric_clean_cols] = df_weather_clean_out[numeric_clean_cols].round(2)
weather_daily_out = os.path.join(PROCESSED_DIR, "daily_weather_processed.csv")
df_weather_clean_out.to_csv(weather_daily_out, index=False)
print(f"Saved cleaned daily weather data to {weather_daily_out}")

Saved daily weather with flags to ../processed\daily_weather_with_flags.csv
Saved cleaned daily weather data to ../processed\daily_weather_processed.csv


In [71]:
# Quick sanity check (daily weather, no resample)
print("Processed weather rows:", len(df_weather_clean_out))
print("With-flags weather rows:", len(df_weather_with_flags))
print("Weather columns (processed):", list(df_weather_clean_out.columns))

Processed weather rows: 365
With-flags weather rows: 365
Weather columns (processed): ['date', 'tavg', 'tmin', 'tmax', 'prcp', 'wspd', 'pres']


# Merge DataSet

In [72]:
aqi_path = os.path.join(PROCESSED_DIR, 'daily_aqi_processed.csv')
weather_path = os.path.join(PROCESSED_DIR, 'daily_weather_processed.csv')
merged_path = os.path.join(PROCESSED_DIR, 'merged_daily.csv')

# Load daily AQI and daily weather data
df_aqi = pd.read_csv(aqi_path).drop(columns=['Unnamed: 0'], errors='ignore')
df_weather = pd.read_csv(weather_path).drop(columns=['Unnamed: 0'], errors='ignore')

# Make sure all date columns are timezone-aware and normalized to midnight
for df in [df_aqi, df_weather]:
    date_col = 'datetime' if 'datetime' in df.columns else 'date'
    df['date'] = pd.to_datetime(df[date_col])
    
    # add or convert timezone
    if df['date'].dt.tz is None:
        df['date'] = df['date'].dt.tz_localize(TIMEZONE)
    else:
        df['date'] = df['date'].dt.tz_convert(TIMEZONE)
    
    df['date'] = df['date'].dt.normalize()
    if date_col != 'date':
        df.drop(columns=[date_col], inplace=True)

# Inner join on date to keep only days present in both datasets
df_merged = pd.merge(df_weather, df_aqi, on='date', how='inner')
os.makedirs(PROCESSED_DIR, exist_ok=True)
df_merged.to_csv(merged_path, index=False)

print(df_merged.head())

                       date  tavg  tmin  tmax  prcp  wspd    pres   pm10  \
0 2023-01-01 00:00:00+07:00  26.5  23.0  31.0   0.6   8.9  1012.7  42.38   
1 2023-01-02 00:00:00+07:00  26.7  23.0  32.0   0.0   6.3  1013.2  51.38   
2 2023-01-03 00:00:00+07:00  28.1  24.0  33.0   1.0   9.7  1012.4  40.10   
3 2023-01-04 00:00:00+07:00  27.6  24.0  33.0   0.1  13.4  1011.7  28.11   
4 2023-01-05 00:00:00+07:00  25.8  24.0  28.0   5.8  17.2  1011.3  16.40   

   pm2_5  nitrogen_dioxide  sulphur_dioxide  ozone  carbon_monoxide  \
0  29.00             42.76            17.00  51.33           710.33   
1  35.54             53.81             9.94  57.96          1090.38   
2  27.82             39.09             5.94  54.12           662.62   
3  19.46             28.41            13.16  46.79           495.75   
4  11.28             21.09            11.22  51.29           352.62   

   uv_index_max  uv_index_clear_sky_max  
0          7.20                    8.80  
1          7.55                 